# Stage 1 of 3 — Mining

**This notebook does ONE thing**: mine `(raw, transpiled)` diff patterns from the
`veerukhannan/mnisq-optbench-pairs` dataset and produce `mined_pairs.jsonl`.
Nothing else runs here — no rule-building, no optimization. That's Stage 2 and
Stage 3, each in their own notebook.

**Setup:**
1. **Add Input** → search `mnisq-optbench-pairs` → Add.
2. **Internet: On** (notebook settings).

**When this finishes:** click **Save Version** ("Save & Run All"). The saved
version's Output tab will contain `mined_pairs.jsonl` — that becomes the input
to Stage 2's notebook (`02_rule_building`), attached there via **Add Input →
Notebook Output → (your username) → this notebook**. That's the actual
checkpoint between stages: a separate file, a separate notebook, not a flag
inside one giant file.

In [ ]:
import glob
print(glob.glob('/kaggle/input/*'))
print(glob.glob('/kaggle/input/**/*', recursive=True)[:20])

In [ ]:
# [mining] extra pulls in pandas/pyarrow (needed here for pd.read_parquet).
!pip install -q "quantum-circuit-smell-intelligence[mining] @ git+https://github.com/veerakrish/quantum-circuit-smell-intelligence.git"

In [ ]:
try:
    import qcs_pipeline
    print("qcs_pipeline imported OK from:", qcs_pipeline.__file__)
except ModuleNotFoundError as e:
    raise RuntimeError(
        "qcs_pipeline is not importable. Check the pip install cell's output "
        "above, or restart the session if you just picked up a code update."
    ) from e

In [ ]:
from pathlib import Path
from qcs_pipeline.mining.from_kaggle_pairs import find_pair_chunks

chunk_files = find_pair_chunks(Path("/kaggle/input"))
DATASET_ROOT = chunk_files[0].parent
print(f"Found {len(chunk_files)} chunk files under {DATASET_ROOT}")
print(chunk_files[:5])

## Confirm the mining parameters against the dataset's own source

The dataset bundles the code that generated it. Read it rather than guess what
`opt_level`, `fidelity_input`/`fidelity_target` mean, or which `opt_level_filter`
value is correct.

In [ ]:
import glob
from pathlib import Path

for filename in ["verify.py", "transpile_pairs.py", "zx_opt.py"]:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        print(f"=== {filename}: NOT FOUND ===\n")
        continue
    path = Path(matches[0])
    print(f"=== {path} ===")
    print(path.read_text())
    print("\n" + "=" * 80 + "\n")

**Confirmed from the source (not assumed):**
- `verify.py`: fidelity is `|⟨a|b⟩|²` from `|0...0⟩`, exact, global-phase-invariant.
- `zx_opt.py`: `opt_level=101` is PyZX's sentinel value, not a real Qiskit level —
  `opt_level_filter=3` below excludes it deliberately, not because it's merely
  the most common value.
- `fidelity_input`/`fidelity_target` are likely each circuit's fidelity against
  an un-inflated base circuit (see `inflation`/`inflate.py`), not input-vs-target
  agreement — a hypothesis, not confirmed (`inflate.py` wasn't printed above).
  Doesn't affect mining either way.

In [ ]:
import pandas as pd

df_peek = pd.read_parquet(chunk_files[0])
print(df_peek.columns.tolist())
print("\nopt_level value counts:")
print(df_peek["opt_level"].value_counts())

In [ ]:
import os
print("CPUs available in this session:", os.cpu_count())

In [ ]:
# The only real work this notebook does. n_workers parallelizes across the
# 120 chunk files -- mining does no quantum simulation (pure QASM parsing +
# difflib diffing), so this is safe, ordinary CPU parallelism.
#
# max_rows_per_chunk=50 is a fast sanity pass (120 x 50 = 6,000 rows max).
# Set to None for the full dataset once everything downstream (Stage 2, 3)
# has been validated on the small pass.
from qcs_pipeline.mining.from_kaggle_pairs import mine_from_parquet

mine_from_parquet(
    dataset_root=DATASET_ROOT,
    out_path=Path("/kaggle/working/mined_pairs.jsonl"),
    opt_level_filter=3,
    max_rows_per_chunk=50,
    n_workers=os.cpu_count(),
)

## Done

`mined_pairs.jsonl` is in `/kaggle/working/`. **Save Version** now ("Save & Run
All") so it becomes this notebook's committed Output — that's what Stage 2
(`02_rule_building.ipynb`) attaches as its input.